In [ ]:
import subprocess
import os, glob
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets

from IPython.display import display
from tqdm import tqdm
from pathlib import Path

In [ ]:
import shutil
print(shutil.which("hd-bet"))

In [ ]:
DATA_ROOT = Path.cwd().parent.parent
PROCESSED_MRI_DIR = DATA_ROOT / "processed_mri"

IMAGES_DIR = DATA_ROOT / "input_mri"

ATLSA_DIR = PROCESSED_MRI_DIR / "atlas_mapping"
LABELS_DIR = PROCESSED_MRI_DIR / "fast"
BET_DIR = PROCESSED_MRI_DIR / "hd_bet"
QC_DIR = PROCESSED_MRI_DIR / "qc"
REGISTRATION_DIR = PROCESSED_MRI_DIR / "registration"
REORIENTED_DIR = PROCESSED_MRI_DIR / "reoriented"

In [ ]:
files = sorted(glob.glob(os.path.join(IMAGES_DIR, "*.nii*")))
f = files[0]  # just test on one file for now
out = os.path.join(BET_DIR, os.path.basename(f))

# proc = subprocess.run(
#     ['hd-bet', '-i', f, '-o', out, '-device', 'cuda', '--verbose'],
#     capture_output=True, text=True
# )

# print(proc.stdout)
# print("--- ERRORS ---")
# print(proc.stderr)
# print("Output exists:", os.path.exists(out))


In [ ]:
#!pip install ipympl
    
%matplotlib widget

In [ ]:
os.makedirs(BET_DIR, exist_ok=True)

files = sorted(glob.glob(os.path.join(IMAGES_DIR, "*.nii*")))

for f in tqdm(files):
    out = os.path.join(BET_DIR, os.path.basename(f))
    if not os.path.exists(out):
        os.system(f'hd-bet -i "{f}" -o "{out}" -device cuda --verbose')

In [ ]:
InitialImage = IMAGES_DIR / "IXI436-HH-2153-T1.nii.gz"
SkullStrippedImage = BET_DIR / "IXI436-HH-2153-T1.nii.gz"
seg = nib.load(LABELS_DIR / "IXI002-Guys-0828-T1_reo_seg.nii.gz").get_fdata()
# Create a color map: 0 = background, 1 = GM (red), 2 = WM (blue)
cmap = {
    0: (0, 0, 0),         # black background
    1: (0.5, 0.5, 0.5),       # gray GM
    2: (1, 1, 1),         # white WM
    3: (0, 0, 0)          # CSF not relevant, considered background
}

# Map seg values to RGB
rgb = np.zeros((*seg.shape, 3))
for label, color in cmap.items():
    rgb[seg == label] = color

# ---- Load your images ----
img_list = [nib.load(p).get_fdata() for p in [InitialImage, SkullStrippedImage]]
img_list.append(rgb)
titles = ["T1", "Skull Stripped T1", "GM - Gray / WM - White Segmentation"]

# Use first image to set initial slice
img = img_list[0]
slice_i = img.shape[2] // 2

# ---- Create subplots ----
n_imgs = len(img_list)
fig, axes = plt.subplots(1, n_imgs, figsize=(5*n_imgs, 5))

# Handle single-axis case
if n_imgs == 1:
    axes = [axes]

# Display initial slices
disps = []
for ax, data, title in zip(axes, img_list, titles):
    disp = ax.imshow(data[:, :, slice_i], cmap="gray")
    disps.append(disp)
    ax.set_title(f"{title} — Slice {slice_i}")
    ax.axis("off")

plt.show()

# ---- Slider ----
slider = widgets.IntSlider(
    value=slice_i,
    min=0,
    max=img.shape[2] - 1,
    description='Slice:',
    continuous_update=True
)

# ---- Update function ----
def update(change):
    i = slider.value
    for disp, ax, data, title in zip(disps, axes, img_list, titles):
        disp.set_data(data[:, :, i])
        ax.set_title(f"{title} — Slice {i}")
    fig.canvas.draw_idle()

slider.observe(update, names="value")
display(slider)

In [ ]:
InitialImage = IMAGES_DIR / "IXI436-HH-2153-T1.nii.gz"
SkullStrippedImage = BET_DIR / "IXI436-HH-2153-T1.nii.gz"

seg = nib.load(LABELS_DIR / "IXI436-HH-2153-T1_seg.nii.gz").get_fdata()

cmap = {
    0: (0, 0, 0),         # background
    1: (0, 0, 0),         # CSF treated as background (for this visualization)
    2: (0.5, 0.5, 0.5),   # GM = gray
    3: (0, 0, 0)          # WM = white
}

rgb_seg = np.zeros((*seg.shape, 3), dtype=np.float32)
for label, color in cmap.items():
    rgb_seg[seg == label] = color

pve_gm = nib.load(LABELS_DIR / "IXI436-HH-2153-T1_pve_1.nii.gz").get_fdata().astype(np.float32)
pve_wm = nib.load(LABELS_DIR / "IXI436-HH-2153-T1_pve_2.nii.gz").get_fdata().astype(np.float32)

pve_mix = np.clip(0.5 * pve_gm + 1.0 * pve_wm, 0.0, 1.0)

# ---- Load images ----
img_list = [
    rgb_seg,        # hard segmentation (RGB)
    pve_gm,         # GM probability (0..1)
    pve_wm,         # WM probability (0..1)
]

titles = [
    "Hard Seg (GM gray / WM white)",
    "PVE GM (0..1)",
    "PVE WM (0..1)",
]

# Use min Z across all volumes (safe if anything differs)
z_max = min(data.shape[2] for data in img_list)
slice_i = z_max // 2

n_imgs = len(img_list)
fig, axes = plt.subplots(1, n_imgs, figsize=(5 * n_imgs, 5))
if n_imgs == 1:
    axes = [axes]

disps = []
for ax, data, title in zip(axes, img_list, titles):
    is_rgb = (data.ndim == 4 and data.shape[-1] == 3)

    if is_rgb:
        disp = ax.imshow(data[:, :, slice_i, :])  # RGB image ignores cmap
    else:
        # For PVE maps, locking vmin/vmax helps consistent contrast
        if "PVE" in title:
            disp = ax.imshow(data[:, :, slice_i], cmap="gray", vmin=0.0, vmax=1.0)
        else:
            disp = ax.imshow(data[:, :, slice_i], cmap="gray")

    disps.append(disp)
    ax.set_title(f"{title} — Slice {slice_i}")
    ax.axis("off")

plt.show()

# ---- Slider ----
slider = widgets.IntSlider(
    value=slice_i,
    min=0,
    max=z_max - 1,
    description="Slice:",
    continuous_update=True
)

# ---- Update function ----
def update(change):
    i = slider.value
    for disp, ax, data, title in zip(disps, axes, img_list, titles):
        is_rgb = (data.ndim == 4 and data.shape[-1] == 3)
        if is_rgb:
            disp.set_data(data[:, :, i, :])
        else:
            disp.set_data(data[:, :, i])
        ax.set_title(f"{title} — Slice {i}")
    fig.canvas.draw_idle()

slider.observe(update, names="value")
display(slider)

In [ ]:
import nibabel as nib
import numpy as np

t1_img   = nib.load( ATLSA_DIR / "IXI002-Guys-0828-T1_reo.nii.gz")
cort_img = nib.load(ATLSA_DIR / "IXI002-Guys-0828-T1_HO_cort_thr25_in_T1.nii.gz")
sub_img  = nib.load(ATLSA_DIR / "IXI002-Guys-0828-T1_HO_sub_thr25_in_T1.nii.gz")
pve_img  = nib.load(LABELS_DIR / "IXI002-Guys-0828-T1_reo_pve_1.nii.gz")

def info(name, img):
    print(name)
    print("  shape:", img.shape)
    print("  axcodes:", nib.aff2axcodes(img.affine))
    print("  affine[0:3, :]:\n", img.affine[:3, :])
    print()

info("T1", t1_img)
info("CORT", cort_img)
info("SUB", sub_img)
info("PVE", pve_img)

print("CORT same grid as T1:", cort_img.shape == t1_img.shape and np.allclose(cort_img.affine, t1_img.affine))
print("SUB  same grid as T1:", sub_img.shape  == t1_img.shape and np.allclose(sub_img.affine,  t1_img.affine))
print("PVE  same grid as T1:", pve_img.shape  == t1_img.shape and np.allclose(pve_img.affine,  t1_img.affine))

In [ ]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from nibabel.processing import resample_from_to
from pathlib import Path
import ipywidgets as widgets

SkullStrippedImage = ATLSA_DIR / "IXI002-Guys-0828-T1_reo.nii.gz"

# ---------- helpers ----------
def load_and_canonical(path, like=None, order=1):
    """
    Load a NIfTI, optionally resample to `like` grid, then reorient to closest canonical (RAS+).
    order=0 for labels (nearest), order=1+ for continuous images.
    """
    img = nib.load(str(path))
    if like is not None:
        img = resample_from_to(img, like, order=order)
    img = nib.as_closest_canonical(img)  # reorders/flips data so voxel axes are canonical (RAS+)
    return img

def axial_for_display(arr3d, k, radiological=False):
    """
    For canonical RAS+ data stored as (x, y, z):
      - Use arr[:, :, k].T so x goes left->right on screen, y goes bottom->top (with origin='lower')
      - Optional radiological convention: flip left/right.
    """
    sl = arr3d[:, :, k].T
    if radiological:
        sl = sl[:, ::-1]  # flip x (L/R) for radiological display
    return sl

def label_cmap(n_labels):
    base = plt.get_cmap("tab20", max(n_labels, 1))
    colors = base(np.arange(max(n_labels, 1)))
    colors[0] = [0, 0, 0, 0]  # transparent background
    return ListedColormap(colors)

def label_norm(n_labels):
    return BoundaryNorm(np.arange(-0.5, n_labels + 0.5, 1), n_labels)

# ---------- load images as NIfTI objects ----------
t1_img   = load_and_canonical(SkullStrippedImage)

cort_img = load_and_canonical(ATLSA_DIR / "IXI002-Guys-0828-T1_HO_cort_thr25_in_T1.nii.gz", like=t1_img, order=0)
sub_img  = load_and_canonical(ATLSA_DIR / "IXI002-Guys-0828-T1_HO_sub_thr25_in_T1.nii.gz",  like=t1_img, order=0)

pve_img  = load_and_canonical(LABELS_DIR / "IXI002-Guys-0828-T1_reo_pve_1.nii.gz", like=t1_img, order=1)

# ---------- to arrays ----------
t1     = t1_img.get_fdata().astype(np.float32)
cort   = cort_img.get_fdata().astype(np.int32)
sub    = sub_img.get_fdata().astype(np.int32)
pve_gm = pve_img.get_fdata().astype(np.float32)

# Optional: restrict cortical labels to likely GM
gm_mask = pve_gm > 0.7
cort_gm = cort * gm_mask

# Intensity normalization for display
lo, hi = np.percentile(t1[t1 > 0], [2, 98])
t1_disp = np.clip((t1 - lo) / (hi - lo + 1e-8), 0, 1)

# Colormaps
n_cort = int(cort_gm.max()) + 1
n_sub  = int(sub.max()) + 1
cmap_cort, norm_cort = label_cmap(n_cort), label_norm(n_cort)
cmap_sub,  norm_sub  = label_cmap(n_sub),  label_norm(n_sub)

# ---------- display ----------
radiological = False  # set True if you want "left on right" style

z_max = min(t1_disp.shape[2], cort_gm.shape[2], sub.shape[2])
slice_i = z_max // 2

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
ax1, ax2 = axes

def draw(i):
    ax1.clear(); ax2.clear()

    base = axial_for_display(t1_disp, i, radiological=radiological)

    cort_sl = axial_for_display(cort_gm, i, radiological=radiological)
    sub_sl  = axial_for_display(sub,     i, radiological=radiological)

    # Mask zeros so background stays transparent even if cmap[0] changes
    cort_sl_m = np.ma.masked_equal(cort_sl, 0)
    sub_sl_m  = np.ma.masked_equal(sub_sl,  0)

    ax1.imshow(base, cmap="gray", origin="lower")
    ax1.imshow(cort_sl_m, cmap=cmap_cort, norm=norm_cort, interpolation="nearest",
               origin="lower", alpha=0.45)
    ax1.set_title(f"Cortical regions (HO) — slice {i}")
    ax1.axis("off")

    ax2.imshow(base, cmap="gray", origin="lower")
    ax2.imshow(sub_sl_m, cmap=cmap_sub, norm=norm_sub, interpolation="nearest",
               origin="lower", alpha=0.55)
    ax2.set_title(f"Subcortical regions (HO) — slice {i}")
    ax2.axis("off")

    fig.canvas.draw_idle()

draw(slice_i)
plt.show()

slider = widgets.IntSlider(value=slice_i, min=0, max=z_max - 1, description="Slice:", continuous_update=True)
def update(change): draw(slider.value)
slider.observe(update, names="value")
display(slider)

In [ ]:
import nibabel as nib
from pathlib import Path
from nibabel.orientations import aff2axcodes

ATLSA_DIR = Path.cwd() / "FastAtlasTest" / "fast_out"

cort_img = nib.load(ATLSA_DIR / "IXI002-Guys-0828-T1_HO_cort_thr25_in_T1.nii.gz")
pve_img  = nib.load(LABELS_DIR / "IXI002-Guys-0828-T1_reo_pve_1.nii.gz")
t1_img   = nib.load(ATLSA_DIR / "IXI002-Guys-0828-T1_reo.nii.gz")  # ideally same one used as -r in applywarp

print("cort:", cort_img.shape, aff2axcodes(cort_img.affine))
print("pve :", pve_img.shape,  aff2axcodes(pve_img.affine))
print("t1  :", t1_img.shape,   aff2axcodes(t1_img.affine))

In [ ]:
import nibabel as nib
import numpy as np

def fast_label_sanity_check(seg_path, pve0_path, pve1_path, pve2_path, n_per_class=10, seed=42):
    # --- load volumes ---
    seg = nib.load(seg_path).get_fdata()
    pve0 = nib.load(pve0_path).get_fdata()
    pve1 = nib.load(pve1_path).get_fdata()
    pve2 = nib.load(pve2_path).get_fdata()

    # --- basic checks ---
    if not (seg.shape == pve0.shape == pve1.shape == pve2.shape):
        raise ValueError(f"Shape mismatch: seg={seg.shape}, pve0={pve0.shape}, pve1={pve1.shape}, pve2={pve2.shape}")

    # seg is often float (because get_fdata), convert safely to int
    seg_i = np.rint(seg).astype(np.int16)

    # Stack PVEs so argmax gives 0=CSF, 1=GM, 2=WM (by FAST convention for -n 3)
    pves = np.stack([pve0, pve1, pve2], axis=-1)  # shape (X,Y,Z,3)
    argmax_map = np.argmax(pves, axis=-1)         # shape (X,Y,Z), values 0..2

    rng = np.random.default_rng(seed)

    label_names = {1: "CSF", 2: "GM", 3: "WM"}

    print("=== Random voxel checks (per hard label) ===")
    for label in (1, 2, 3):
        coords = np.argwhere(seg_i == label)
        if coords.size == 0:
            print(f"\nLabel {label} ({label_names[label]}): no voxels found in seg.")
            continue

        k = min(n_per_class, len(coords))
        chosen = coords[rng.choice(len(coords), size=k, replace=False)]

        # For each chosen voxel, check which PVE is largest
        correct = 0
        print(f"\nLabel {label} ({label_names[label]}) — sampled {k} voxels:")
        for (x, y, z) in chosen:
            pv = pves[x, y, z, :]  # [pve0,pve1,pve2]
            am = int(np.argmax(pv))  # 0..2
            expected = label - 1     # 1->0, 2->1, 3->2
            ok = (am == expected)
            correct += int(ok)

            print(
                f"  voxel=({x:3d},{y:3d},{z:3d})  "
                f"pves=[{pv[0]:.3f},{pv[1]:.3f},{pv[2]:.3f}]  "
                f"argmax={am} ({['CSF','GM','WM'][am]})  "
                f"expected={expected} ({label_names[label]})  "
                f"{'OK' if ok else 'MISMATCH'}"
            )

        print(f"  -> agreement in sample: {correct}/{k} = {correct/k:.1%}")

    # --- Overall agreement (more robust than sampling) ---
    brain_mask = seg_i > 0  # exclude background
    expected_map = seg_i - 1  # 1..3 -> 0..2
    overall = np.mean((argmax_map[brain_mask] == expected_map[brain_mask]))

    print("\n=== Overall agreement across seg>0 voxels ===")
    print(f"Agreement = {overall:.2%} (higher is better; expect very high if everything matches)")

    # Optional: confusion matrix (counts)
    # rows = hard seg class (expected), cols = argmax class
    cm = np.zeros((3, 3), dtype=np.int64)
    em = expected_map[brain_mask].astype(int)
    am = argmax_map[brain_mask].astype(int)
    for i in range(3):
        for j in range(3):
            cm[i, j] = np.sum((em == i) & (am == j))

    print("\nConfusion matrix (rows=seg class, cols=argmax pve class) with classes [CSF, GM, WM]:")
    print(cm)

# ---- Example call with your typical naming ----
base = "IXI313-HH-2241-T1"  # change as needed

seg_path  = LABELS_DIR / f"{base}_seg.nii.gz"
pve0_path = LABELS_DIR / f"{base}_pve_0.nii.gz"
pve1_path = LABELS_DIR / f"{base}_pve_1.nii.gz"
pve2_path = LABELS_DIR / f"{base}_pve_2.nii.gz"

fast_label_sanity_check(seg_path, pve0_path, pve1_path, pve2_path, n_per_class=10, seed=42)

In [ ]:
import nibabel as nib
import numpy as np

def confusion(cm_expected, cm_pred, n_classes=3):
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for i in range(n_classes):
        for j in range(n_classes):
            cm[i, j] = np.sum((cm_expected == i) & (cm_pred == j))
    return cm

def eval_fast_seg_vs_pve(seg_path, pve0_path, pve1_path, pve2_path, thresholds=(0.5,0.6,0.7,0.8,0.9), margin_list=(0.0,0.1,0.2,0.3)):
    seg = np.rint(nib.load(seg_path).get_fdata()).astype(np.int16)
    pve0 = nib.load(pve0_path).get_fdata().astype(np.float32)
    pve1 = nib.load(pve1_path).get_fdata().astype(np.float32)
    pve2 = nib.load(pve2_path).get_fdata().astype(np.float32)

    if not (seg.shape == pve0.shape == pve1.shape == pve2.shape):
        raise ValueError("Shape mismatch between seg and PVEs")

    # expected: seg 1..3 -> 0..2 (CSF, GM, WM)
    brain = seg > 0
    expected = (seg - 1)[brain].astype(np.int16)

    pves = np.stack([pve0, pve1, pve2], axis=-1)[brain]   # (N,3)
    pred = np.argmax(pves, axis=-1).astype(np.int16)

    # confidence stats
    top1 = np.max(pves, axis=-1)
    # top2: second highest
    part = np.partition(pves, -2, axis=-1)
    top2 = part[:, -2]
    margin = top1 - top2

    def report(mask, label):
        n = int(mask.sum())
        if n == 0:
            print(f"\n[{label}] kept 0 voxels")
            return
        cm = confusion(expected[mask], pred[mask], 3)
        acc = np.trace(cm) / cm.sum()
        print(f"\n[{label}] kept {n:,} voxels ({n/expected.size:.1%} coverage)")
        print(f"agreement: {acc:.2%}")
        print("confusion (rows=seg, cols=argmax PVE) classes [CSF, GM, WM]:")
        print(cm)

    # baseline (no filtering besides brain mask)
    report(np.ones_like(expected, dtype=bool), "baseline (all seg>0 voxels)")

    # threshold filtering
    for t in thresholds:
        report(top1 >= t, f"max(PVE) >= {t}")

    # margin filtering (optionally combine with a moderate top1 threshold)
    for m in margin_list:
        report(margin >= m, f"(top1-top2) >= {m}")

    # combined example (often a sweet spot)
    report((top1 >= 0.7) & (margin >= 0.2), "combined: top1>=0.7 & margin>=0.2")


# Example call
base = "IXI313-HH-2241-T1"
seg_path  = LABELS_DIR / f"{base}_seg.nii.gz"
pve0_path = LABELS_DIR / f"{base}_pve_0.nii.gz"
pve1_path = LABELS_DIR / f"{base}_pve_1.nii.gz"
pve2_path = LABELS_DIR / f"{base}_pve_2.nii.gz"

eval_fast_seg_vs_pve(seg_path, pve0_path, pve1_path, pve2_path)